# CS5: Capital Controls & Exchange Rate Regimes Analysis

**Purpose**: Analyze the relationship between financial openness, capital controls, exchange rate regimes, and capital flow volatility using external data sources.

**Source Code Extraction**: `src/dashboard/reports/cs5_report.py`
- Lines 153-154: Pearson correlation for capital controls
- Lines 361-367: F-test calculations for regime analysis
- Lines 231-234: Country aggregate correlations

**External Data Sources**:
- Capital Controls: Fernández et al. (2016) Capital Control Measures Database (1999-2017)
- Exchange Rate Regimes: Ilzetzki, Reinhart, and Rogoff (2019) Classification (1999-2019)

## Section 1: Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add stats_core to path
sys.path.append('../lib')
from stats_core import calculate_f_statistic, get_significance_stars

print("CS5 Analysis: Capital Controls & Exchange Rate Regimes")
print("="*60)

## Section 2: Load Capital Controls Data

In [ ]:
# Load capital controls data
controls_path = Path('../data/CS5_Capital_Controls')

# Load yearly standard deviations with capital control indices
yearly_sd = pd.read_csv(controls_path / 'sd_yearly_flows.csv')
yearly_sd_no_outliers = pd.read_csv(controls_path / 'sd_yearly_flows_no_outliers.csv')

# Load country aggregate standard deviations
country_sd = pd.read_csv(controls_path / 'sd_country_flows.csv')
country_sd_no_outliers = pd.read_csv(controls_path / 'sd_country_flows_no_outliers.csv')

print(f"Loaded capital controls data for {len(yearly_sd['COUNTRY'].unique())} countries")
print(f"Years covered: {yearly_sd['YEAR'].min()}-{yearly_sd['YEAR'].max()}")
print(f"\nIndicators available:")
for col in yearly_sd.columns:
    if 'restrictions' in col.lower():
        print(f"  - {col}")

## Section 3: Capital Controls Correlation Analysis (Yearly Data)

In [ ]:
def calculate_capital_controls_correlation(data, indicator='yearly_sd_net_capital_flows_pgdp', 
                                          control_index='mean_overall_restrictions_index'):
    """
    Calculate correlation between capital controls and flow volatility.
    Source: Extracted from cs5_report.py lines 153-154
    """
    # Remove NaN values
    df_clean = data.dropna(subset=[indicator, control_index])
    
    if len(df_clean) < 3:
        return {'correlation': np.nan, 'p_value': np.nan, 'n_obs': len(df_clean)}
    
    # Calculate Pearson correlation
    corr, p_value = stats.pearsonr(
        df_clean[control_index],
        df_clean[indicator]
    )
    
    return {
        'correlation': corr,
        'p_value': p_value,
        'n_obs': len(df_clean),
        'significance': get_significance_stars(p_value)
    }

In [ ]:
# Analyze yearly correlations for different indicators
print("\n" + "="*60)
print("YEARLY CAPITAL CONTROLS CORRELATIONS")
print("="*60)

flow_indicators = [
    'yearly_sd_net_capital_flows_pgdp',
    'yearly_sd_net_direct_investment_pgdp',
    'yearly_sd_net_portfolio_investment_pgdp',
    'yearly_sd_net_other_investment_pgdp'
]

yearly_results = []
for indicator in flow_indicators:
    if indicator in yearly_sd.columns:
        # Full data
        result_full = calculate_capital_controls_correlation(yearly_sd, indicator)
        # No outliers
        result_no_outliers = calculate_capital_controls_correlation(yearly_sd_no_outliers, indicator)
        
        indicator_name = indicator.replace('yearly_sd_', '').replace('_pgdp', '').replace('_', ' ').title()
        
        print(f"\n{indicator_name}:")
        print(f"  Full data: Corr = {result_full['correlation']:.4f}, P-value = {result_full['p_value']:.4f} {result_full['significance']}")
        print(f"  No outliers: Corr = {result_no_outliers['correlation']:.4f}, P-value = {result_no_outliers['p_value']:.4f} {result_no_outliers['significance']}")
        
        yearly_results.append({
            'Indicator': indicator_name,
            'Full_Correlation': result_full['correlation'],
            'Full_PValue': result_full['p_value'],
            'Full_Sig': result_full['significance'],
            'NoOutliers_Correlation': result_no_outliers['correlation'],
            'NoOutliers_PValue': result_no_outliers['p_value'],
            'NoOutliers_Sig': result_no_outliers['significance']
        })

## Section 4: Country Aggregate Correlations

In [ ]:
# Country-level aggregate analysis
print("\n" + "="*60)
print("COUNTRY AGGREGATE CAPITAL CONTROLS CORRELATIONS")
print("="*60)

country_indicators = [
    'country_sd_net_capital_flows_pgdp',
    'country_sd_net_direct_investment_pgdp',
    'country_sd_net_portfolio_investment_pgdp',
    'country_sd_net_other_investment_pgdp'
]

country_results = []
for indicator in country_indicators:
    if indicator in country_sd.columns:
        # Full data
        result_full = calculate_capital_controls_correlation(country_sd, indicator)
        # No outliers  
        result_no_outliers = calculate_capital_controls_correlation(country_sd_no_outliers, indicator)
        
        indicator_name = indicator.replace('country_sd_', '').replace('_pgdp', '').replace('_', ' ').title()
        
        print(f"\n{indicator_name}:")
        print(f"  Full data: Corr = {result_full['correlation']:.4f}, P-value = {result_full['p_value']:.4f} {result_full['significance']}")
        print(f"  No outliers: Corr = {result_no_outliers['correlation']:.4f}, P-value = {result_no_outliers['p_value']:.4f} {result_no_outliers['significance']}")
        
        country_results.append({
            'Indicator': indicator_name,
            'Full_Correlation': result_full['correlation'],
            'Full_PValue': result_full['p_value'],
            'Full_Sig': result_full['significance'],
            'NoOutliers_Correlation': result_no_outliers['correlation'],
            'NoOutliers_PValue': result_no_outliers['p_value'],
            'NoOutliers_Sig': result_no_outliers['significance']
        })

## Section 5: Iceland-Specific Analysis

In [ ]:
# Extract Iceland-specific data
iceland_yearly = yearly_sd[yearly_sd['COUNTRY'] == 'Iceland']
iceland_country = country_sd[country_sd['COUNTRY'] == 'Iceland']

print("\n" + "="*60)
print("ICELAND CAPITAL CONTROLS PROFILE")
print("="*60)

if not iceland_country.empty:
    print(f"\nCountry-level averages (1999-2017):")
    print(f"  Overall Restrictions Index: {iceland_country['mean_overall_restrictions_index'].values[0]:.4f}")
    print(f"  Net Capital Flows Volatility: {iceland_country['country_sd_net_capital_flows_pgdp'].values[0]:.4f}")

if not iceland_yearly.empty:
    print(f"\nYearly statistics:")
    print(f"  Years with data: {len(iceland_yearly)}")
    print(f"  Mean restrictions index: {iceland_yearly['mean_overall_restrictions_index'].mean():.4f}")
    print(f"  Min restrictions index: {iceland_yearly['mean_overall_restrictions_index'].min():.4f}")
    print(f"  Max restrictions index: {iceland_yearly['mean_overall_restrictions_index'].max():.4f}")
    
    # Find years with highest/lowest controls
    max_control_year = iceland_yearly.loc[iceland_yearly['mean_overall_restrictions_index'].idxmax(), 'YEAR']
    min_control_year = iceland_yearly.loc[iceland_yearly['mean_overall_restrictions_index'].idxmin(), 'YEAR']
    print(f"  Year with highest controls: {int(max_control_year)}")
    print(f"  Year with lowest controls: {int(min_control_year)}")

## Section 6: Load Exchange Rate Regime Data

In [ ]:
# Load exchange rate regime analysis data
regime_path = Path('../data/CS5_Regime_Analysis')

regime_indicators = {
    'Net Capital Flows': 'net_capital_flows',
    'Net Direct Investment': 'net_direct_investment',
    'Net Portfolio Investment': 'net_portfolio_investment',
    'Net Other Investment': 'net_other_investment'
}

regime_data = {}
for indicator_name, file_prefix in regime_indicators.items():
    regime_data[indicator_name] = {
        'full': pd.read_csv(regime_path / f"{file_prefix}_full.csv"),
        'no_crises': pd.read_csv(regime_path / f"{file_prefix}_no_crises.csv")
    }

print(f"\nLoaded exchange rate regime data for {len(regime_indicators)} indicators")
print("\nExchange rate regime categories:")
sample_df = regime_data['Net Capital Flows']['full']
regime_cols = [col for col in sample_df.columns if '_pgdp' in col and 'iceland' not in col]
for col in regime_cols:
    regime_name = col.replace('_pgdp_weighted', '').replace('_pgdp_simple', '').replace('_pgdp', '').replace('_', ' ').title()
    if 'weighted' not in col and 'simple' not in col:
        print(f"  - {regime_name}")

## Section 7: Exchange Rate Regime F-Test Implementation

In [ ]:
def calculate_regime_f_tests(regime_data, indicator, include_crisis=True):
    """
    Calculate F-tests comparing Iceland to different exchange rate regimes.
    Source: Extracted from cs5_report.py lines 361-367
    """
    data_key = 'full' if include_crisis else 'no_crises'
    df = regime_data[indicator][data_key]
    
    # Calculate Iceland standard deviation
    iceland_data = df['iceland_pgdp'].dropna()
    iceland_std = np.std(iceland_data, ddof=1) if len(iceland_data) > 1 else np.nan
    iceland_n = len(iceland_data)
    
    results = []
    
    # Define regime groups (using weighted averages)
    regime_groups = {
        'Hard Peg': 'hard_peg_pgdp_weighted',
        'Crawling/Tight': 'crawl_tight_pgdp_weighted',
        'Managed Float': 'managed_float_pgdp_weighted',
        'Free Float': 'free_float_pgdp_weighted',
        'Freely Falling': 'freely_falling_pgdp_weighted',
        'Free/Managed': 'free_managed_pgdp_weighted'
    }
    
    for regime_name, regime_col in regime_groups.items():
        if regime_col in df.columns:
            regime_data = df[regime_col].dropna()
            
            if len(regime_data) > 1:
                regime_std = np.std(regime_data, ddof=1)
                regime_n = len(regime_data)
                
                # F-test calculation (from cs5_report.py lines 363-367)
                if regime_std > 0 and iceland_std > 0:
                    f_stat = iceland_std**2 / regime_std**2
                    df1 = iceland_n - 1
                    df2 = regime_n - 1
                    p_value = 2 * min(stats.f.cdf(f_stat, df1, df2), 
                                     1 - stats.f.cdf(f_stat, df1, df2))
                else:
                    f_stat = np.nan
                    p_value = np.nan
            else:
                regime_std = np.nan
                f_stat = np.nan
                p_value = np.nan
                regime_n = len(regime_data)
                
            results.append({
                'Regime': regime_name,
                'Regime_StdDev': regime_std,
                'Regime_N': regime_n,
                'Iceland_StdDev': iceland_std,
                'Iceland_N': iceland_n,
                'F_Statistic': f_stat,
                'P_Value': p_value,
                'Significance': get_significance_stars(p_value) if not np.isnan(p_value) else ''
            })
    
    return pd.DataFrame(results)

## Section 8: Regime Analysis - Full Period

In [ ]:
# Analyze all indicators for full period
print("\n" + "="*60)
print("EXCHANGE RATE REGIME F-TESTS (FULL PERIOD)")
print("="*60)

regime_f_test_results_full = []

for indicator_name in regime_indicators.keys():
    print(f"\n{indicator_name}:")
    print("-" * 40)
    
    results = calculate_regime_f_tests(regime_data, indicator_name, include_crisis=True)
    
    for _, row in results.iterrows():
        print(f"  {row['Regime']:15} | Iceland SD: {row['Iceland_StdDev']:.4f} | Regime SD: {row['Regime_StdDev']:.4f}")
        print(f"  {'':15} | F-stat: {row['F_Statistic']:.4f} | P-value: {row['P_Value']:.4f} {row['Significance']}")
    
    # Add indicator column for combined results
    results['Indicator'] = indicator_name
    results['Period'] = 'Full'
    regime_f_test_results_full.append(results)

# Combine all results
all_regime_results_full = pd.concat(regime_f_test_results_full, ignore_index=True)

## Section 9: Regime Analysis - Crisis Excluded

In [ ]:
# Analyze all indicators for crisis-excluded period
print("\n" + "="*60)
print("EXCHANGE RATE REGIME F-TESTS (CRISIS EXCLUDED)")
print("="*60)

regime_f_test_results_crisis_excluded = []

for indicator_name in regime_indicators.keys():
    print(f"\n{indicator_name}:")
    print("-" * 40)
    
    results = calculate_regime_f_tests(regime_data, indicator_name, include_crisis=False)
    
    for _, row in results.iterrows():
        print(f"  {row['Regime']:15} | Iceland SD: {row['Iceland_StdDev']:.4f} | Regime SD: {row['Regime_StdDev']:.4f}")
        print(f"  {'':15} | F-stat: {row['F_Statistic']:.4f} | P-value: {row['P_Value']:.4f} {row['Significance']}")
    
    # Add indicator column for combined results
    results['Indicator'] = indicator_name
    results['Period'] = 'Crisis-Excluded'
    regime_f_test_results_crisis_excluded.append(results)

# Combine all results
all_regime_results_crisis_excluded = pd.concat(regime_f_test_results_crisis_excluded, ignore_index=True)

## Section 10: Summary Statistics

In [ ]:
# Summary of significant findings
print("\n" + "="*60)
print("SUMMARY OF SIGNIFICANT FINDINGS")
print("="*60)

# Capital Controls Summary
print("\nCapital Controls Correlations:")
yearly_df = pd.DataFrame(yearly_results)
country_df = pd.DataFrame(country_results)

print("\nYearly Analysis:")
for _, row in yearly_df.iterrows():
    if row['Full_Sig'] != '':
        print(f"  {row['Indicator']}: r = {row['Full_Correlation']:.3f} {row['Full_Sig']}")

print("\nCountry Aggregate Analysis:")
for _, row in country_df.iterrows():
    if row['Full_Sig'] != '':
        print(f"  {row['Indicator']}: r = {row['Full_Correlation']:.3f} {row['Full_Sig']}")

# Exchange Rate Regime Summary
print("\nExchange Rate Regime F-Tests (Full Period):")
significant_regime_full = all_regime_results_full[all_regime_results_full['Significance'] != '']
print(f"  Total significant comparisons: {len(significant_regime_full)} / {len(all_regime_results_full)}")

# Count by significance level
for sig_level in ['***', '**', '*']:
    count = len(significant_regime_full[significant_regime_full['Significance'] == sig_level])
    if count > 0:
        print(f"  At {sig_level} level: {count} comparisons")

print("\nExchange Rate Regime F-Tests (Crisis Excluded):")
significant_regime_excluded = all_regime_results_crisis_excluded[all_regime_results_crisis_excluded['Significance'] != '']
print(f"  Total significant comparisons: {len(significant_regime_excluded)} / {len(all_regime_results_crisis_excluded)}")

## Section 11: Save Results

In [ ]:
# Save all results to CSV
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# Save capital controls correlations
yearly_df.to_csv(output_dir / 'CS5_capital_controls_yearly.csv', index=False)
country_df.to_csv(output_dir / 'CS5_capital_controls_country.csv', index=False)

# Save regime F-test results
all_regime_results_full.to_csv(output_dir / 'CS5_regime_ftests_full.csv', index=False)
all_regime_results_crisis_excluded.to_csv(output_dir / 'CS5_regime_ftests_crisis_excluded.csv', index=False)

# Create summary table
summary = {
    'Analysis': ['Capital Controls (Yearly)', 'Capital Controls (Country)', 
                 'Exchange Rate Regimes (Full)', 'Exchange Rate Regimes (Crisis-Excluded)'],
    'N_Tests': [len(yearly_df) * 2, len(country_df) * 2,  # *2 for full and no outliers
               len(all_regime_results_full), len(all_regime_results_crisis_excluded)],
    'N_Significant': [
        sum((yearly_df['Full_Sig'] != '') | (yearly_df['NoOutliers_Sig'] != '')),
        sum((country_df['Full_Sig'] != '') | (country_df['NoOutliers_Sig'] != '')),
        len(significant_regime_full),
        len(significant_regime_excluded)
    ]
}
summary_df = pd.DataFrame(summary)
summary_df.to_csv(output_dir / 'CS5_analysis_summary.csv', index=False)

print("\n" + "="*60)
print("RESULTS SAVED")
print("="*60)
print(f"Results saved to {output_dir}/")
print("  - CS5_capital_controls_yearly.csv")
print("  - CS5_capital_controls_country.csv")
print("  - CS5_regime_ftests_full.csv")
print("  - CS5_regime_ftests_crisis_excluded.csv")
print("  - CS5_analysis_summary.csv")

## Section 12: Final Report

In [ ]:
print("\n" + "="*60)
print("CS5 ANALYSIS COMPLETE")
print("="*60)

print("\nAnalysis Components:")
print("1. Capital Controls Correlation Analysis (1999-2017)")
print(f"   - Yearly correlations: {len(yearly_df)} indicators analyzed")
print(f"   - Country aggregate correlations: {len(country_df)} indicators analyzed")
print(f"   - External data source: Fernández et al. (2016)")

print("\n2. Exchange Rate Regime Analysis (1999-2019)")
print(f"   - Regimes compared: 6 categories")
print(f"   - Indicators analyzed: {len(regime_indicators)}")
print(f"   - F-tests performed: {len(all_regime_results_full) + len(all_regime_results_crisis_excluded)} total")
print(f"   - External data source: Ilzetzki, Reinhart, and Rogoff (2019)")

print("\nKey Findings:")
print(f"  - Capital controls show {'positive' if yearly_df['Full_Correlation'].mean() > 0 else 'negative'} "
      f"average correlation with volatility")
print(f"  - Iceland volatility significantly different from {len(significant_regime_full)} "
      f"regime comparisons (full period)")
print(f"  - Crisis exclusion affects {abs(len(significant_regime_full) - len(significant_regime_excluded))} "
      f"significance results")

print("\n" + "="*60)